# Patent Prior Art Multimodal RAG System

This notebook demonstrates the end-to-end functionality of the Patent Prior Art RAG System, which determines whether a new invention description conflicts with existing patents or prior art.

We reuse token-aware chunking, batch embeddings (ChromaDB), semantic retrieval, and LLM reasoning.

In [ ]:
import os
import sys
import logging

# Add the parent directory to sys.path so we can import from patent_rag
sys.path.append(os.path.abspath('..'))

# Load environment variables from the parent directory's .env file
from dotenv import load_dotenv
env_path = os.path.join(os.path.abspath('..'), '.env')
load_dotenv(env_path)

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

# Ensure GEMINI API key is set
if 'GEMINI_API_KEY' not in os.environ:
    print("Please set GEMINI_API_KEY environment variable in the .env file.")
else:
    print("Environment and paths configured correctly.")

### 1. Initialize Pipeline Components
We initialize data ingestion, document processing, embedding, and vector storage.

In [ ]:
from data_ingestion import DataIngestionPipeline
from document_processor import PatentChunker
from embedding_pipeline import EmbeddingPipeline
from vector_store import PatentVectorStore
from retrieval_engine import PatentRetrievalEngine
from novelty_analyzer import NoveltyAnalyzer

ingestion = DataIngestionPipeline()
documents = ingestion.ingest_all()

chunker = PatentChunker(chunk_size_tokens=500, chunk_overlap_tokens=100)
chunks = chunker.chunk_documents(documents)

embedder = EmbeddingPipeline(model_name="all-MiniLM-L6-v2")
vector_store = PatentVectorStore(embedder=embedder)
vector_store.add_chunks(chunks, batch_size=32)

retriever = PatentRetrievalEngine(vector_store=vector_store)
analyzer = NoveltyAnalyzer()

### 2. Run Example Query
We query the system with a novel invention description and observe the results.

In [ ]:
invention_desc = "An AI-based drone navigation system using LiDAR sensors and deep learning for real-time obstacle avoidance."
print(f"Invention Description: {invention_desc}\n")

results = retriever.retrieve_similar_documents(invention_desc, top_k=2)
analysis_report = analyzer.analyze(invention_desc, results)
print("NOVELTY ANALYSIS REPORT:\n")
print(analysis_report)

### 3. Evaluate Results
We can plot the similarity distributions.

In [ ]:
from evaluation import EvaluationModule

evaluator = EvaluationModule()
all_retrieved = results['similar_patents'] + results['similar_papers'] + results['similar_claims']
evaluator.plot_similarity_distribution(all_retrieved, output_path="demo_similarity.png")

from IPython.display import Image, display
if os.path.exists("demo_similarity.png"):
    display(Image(filename="demo_similarity.png"))